In [2]:
import os
import numpy as np
import cv2
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# untar
!ls
!tar -xvzf dataset.tar.gz
# load train
train_images = pickle.load(open('train_images.pkl', 'rb'))
train_labels = pickle.load(open('train_labels.pkl', 'rb'))
# load val
val_images = pickle.load(open('val_images.pkl', 'rb'))
val_labels = pickle.load(open('val_labels.pkl', 'rb'))

'ls' is not recognized as an internal or external command,
operable program or batch file.
x train_images.pkl
x train_labels.pkl
x val_images.pkl
x val_labels.pkl


In [5]:
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)

train_images = train_images.permute(0, 3, 1, 2)
val_images = val_images.permute(0, 3, 1, 2)

train_dataset = TensorDataset(train_images,
                              torch.tensor(train_labels.squeeze(), dtype=torch.long))
val_dataset = TensorDataset(val_images,
                            torch.tensor(val_labels.squeeze(), dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [6]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        self.model = nn.Sequential(
            # First block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Second block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten layer
            nn.Flatten(),

            # Fully connected block: Dense -> ReLU -> Dropout -> Dense -> Softmax
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [7]:
model = ConvNet()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

In [8]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()  # Zero the parameter gradients
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update tqdm description with current loss and accuracy
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

In [9]:
def validate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the validation loop
    val_loader_tqdm = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():  # Disable gradient calculations for validation
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Update tqdm description with current validation loss and accuracy
            val_loader_tqdm.set_postfix(loss=val_loss / total, accuracy=100 * correct / total)

    val_accuracy = 100 * correct / total
    val_loss = val_loss / len(val_loader)
    return val_loss, val_accuracy

## **Start modified network pruning**

In [108]:
# reload full model
model = ConvNet().to(device)
model.load_state_dict(torch.load('original_model.pt'))
print("Loaded original model weights.")


Loaded original model weights.


C:\Users\nickc\AppData\Local\Temp\ipykernel_62068\3605758991.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('original_model.pt'))


In [109]:
# class SlimmableConvNet(nn.Module):
#     def __init__(self, linear_input_dim=1024):
#         super(SlimmableConvNet, self).__init__()

#         self.model = nn.Sequential(
#             nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2),
#             nn.Dropout(0.25),

#             nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2),
#             nn.Dropout(0.25),

#             nn.Flatten(),
#             nn.Linear(linear_input_dim, 512),
#             nn.ReLU(),
#             nn.Dropout(0.5),
#             nn.Linear(512, 5),
#         )

#     def forward(self, x):
#         return self.model(x)


In [110]:
class SlimmableConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [111]:
def get_bn_scaling_factors(model):
    gamma_list = []
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            gamma_list.append(m.weight.detach().abs())
    return torch.cat(gamma_list)

In [112]:
def get_bn_masks(model, prune_ratio):
    masks = []
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            gamma = m.weight.detach().abs()
            threshold = torch.quantile(gamma, prune_ratio)
            mask = gamma > threshold
            masks.append(mask)
    return masks

In [113]:
def transfer_weights(model_with_bn, model_no_bn, masks):
    model_with_bn.eval()
    model_no_bn.eval()

    device = next(model_with_bn.parameters()).device

    layers_bn = [m for m in model_with_bn.model if isinstance(m, (nn.Conv2d, nn.Linear))]
    layers_no_bn = [m for m in model_no_bn.model if isinstance(m, (nn.Conv2d, nn.Linear))]

    prev_mask = torch.ones(3, dtype=torch.bool)  # Start on CPU for indexing
    mask_idx = 0

    for i, (m_bn, m_no_bn) in enumerate(zip(layers_bn, layers_no_bn)):
        if isinstance(m_bn, nn.Conv2d):
            out_mask = masks[mask_idx].to('cpu')
            prev_mask = prev_mask.to('cpu')

            W_src = m_bn.weight.data.cpu()
            W_dst = m_no_bn.weight.data

            # Copy over weights for kept channels
            for out_i, keep_out in enumerate(out_mask):
                if not keep_out:
                    continue
                for in_i, keep_in in enumerate(prev_mask):
                    if keep_in:
                        W_dst[out_i, in_i] = W_src[out_i, in_i]

            # Copy bias
            if m_bn.bias is not None:
                B_src = m_bn.bias.data.cpu()
                B_dst = m_no_bn.bias.data
                for out_i, keep_out in enumerate(out_mask):
                    if keep_out:
                        B_dst[out_i] = B_src[out_i]

            prev_mask = out_mask
            mask_idx += 1

        elif isinstance(m_bn, nn.Linear):
            m_no_bn.weight.data.copy_(m_bn.weight.data.clone())
            m_no_bn.bias.data.copy_(m_bn.bias.data.clone())


In [114]:
model_with_bn = SlimmableConvNet().to(device)
model_with_bn.load_state_dict(torch.load("slim_model_with_bn.pth"))
model_with_bn.eval()

C:\Users\nickc\AppData\Local\Temp\ipykernel_62068\670936557.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_with_bn.load_state_dict(torch.load("slim_model_with_bn.

SlimmableConvNet(
  (model): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.25, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (15): Dropout(p=0.25, inplace=False)
    (16): Flatten(

In [115]:
prune_ratio = 0.22  # Prune 30% of BN channels
masks = get_bn_masks(model_with_bn, prune_ratio)

In [116]:
model_with_bn = SlimmableConvNet()
model_with_bn.load_state_dict(torch.load("slim_model_with_bn.pth"))
model_with_bn.eval()

C:\Users\nickc\AppData\Local\Temp\ipykernel_62068\4247666759.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_with_bn.load_state_dict(torch.load("slim_model_with_bn

SlimmableConvNet(
  (model): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.25, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (15): Dropout(p=0.25, inplace=False)
    (16): Flatten(

In [117]:
#model_no_bn = ConvNet().to(device)

model_no_bn = ConvNet().to(device)
model_no_bn.load_state_dict(torch.load("original_model.pt"))
model_no_bn.eval()

transfer_weights(model_with_bn, model_no_bn, masks)

C:\Users\nickc\AppData\Local\Temp\ipykernel_62068\2664345614.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_no_bn.load_state_dict(torch.load("original_model.pt"))

In [118]:
# Apply sparsity mask immediately
with torch.no_grad():
    mask_idx = 0
    prev_mask = torch.ones(3, dtype=torch.bool)
    for m in model_no_bn.model:
        if isinstance(m, nn.Conv2d):
            out_mask = masks[mask_idx].to('cpu')
            prev_mask = prev_mask.to('cpu')

            for out_i in range(m.weight.shape[0]):
                for in_i in range(m.weight.shape[1]):
                    if not (out_mask[out_i] and prev_mask[in_i]):
                        m.weight.data[out_i, in_i].zero_()

            if m.bias is not None:
                for out_i in range(m.bias.shape[0]):
                    if not out_mask[out_i]:
                        m.bias.data[out_i] = 0.0

            prev_mask = out_mask
            mask_idx += 1


In [119]:
val_loss, val_acc = validate(model_no_bn, val_loader, criterion, device)
print(f"Pruned No-BN Model Accuracy: {val_acc:.4f}, Loss: {val_loss:.4f}")

Pruned No-BN Model Accuracy: 21.5446, Loss: 1.5978


In [120]:
print("== Sanity Check: Conv1 Weights ==")
w_bn = model_with_bn.model[0].weight.data
w_nobn = model_no_bn.model[0].weight.data
print(f"BN model weight mean: {w_bn.mean():.4f}, std: {w_bn.std():.4f}")
print(f"NoBN model weight mean: {w_nobn.mean():.4f}, std: {w_nobn.std():.4f}")
print(f"NoBN nonzero weights: {(w_nobn != 0).sum().item()} / {w_nobn.numel()}")


== Sanity Check: Conv1 Weights ==
BN model weight mean: -0.0028, std: 0.1130
NoBN model weight mean: -0.0008, std: 0.0999
NoBN nonzero weights: 675 / 864


## **Fine Tuning**

In [121]:
weight_masks = {}
for name, param in model_no_bn.named_parameters():
    if "weight" in name:
        mask = (param != 0).float()
        weight_masks[name] = mask.to(param.device)


In [122]:
def train_one_epoch_with_mask(model, train_loader, optimizer, criterion, device, weight_masks):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Mask gradients
        for name, param in model.named_parameters():
            if name in weight_masks and param.grad is not None:
                param.grad *= weight_masks[name]

        # Optimizer step
        optimizer.step()

        # Optional: enforce masking on weights again (in case of numerical drift)
        for name, param in model.named_parameters():
            if name in weight_masks:
                param.data *= weight_masks[name]

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy


In [123]:
optimizer = torch.optim.Adam(model_no_bn.parameters(), lr=1e-5, weight_decay=1e-6)

# fine-tuning epochs
for epoch in range(8):
    train_loss, train_accuracy = train_one_epoch_with_mask(model_no_bn, train_loader, optimizer, criterion, device, weight_masks)
    val_loss, val_accuracy = validate(model_no_bn, val_loader, criterion, device)
    print(f'[Fine-Tuning Epoch {epoch+1}] Val Accuracy: {val_accuracy:.2f}%')

[Fine-Tuning Epoch 1] Val Accuracy: 48.40%


[Fine-Tuning Epoch 2] Val Accuracy: 53.43%


[Fine-Tuning Epoch 3] Val Accuracy: 56.75%


[Fine-Tuning Epoch 4] Val Accuracy: 58.18%


[Fine-Tuning Epoch 5] Val Accuracy: 58.65%


[Fine-Tuning Epoch 6] Val Accuracy: 59.56%


[Fine-Tuning Epoch 7] Val Accuracy: 58.61%


[Fine-Tuning Epoch 8] Val Accuracy: 60.44%


In [124]:
def report_sparsity_and_accuracy(model, val_loader, criterion, device):
    model.eval()
    
    total_weights = 0
    zero_weights = 0

    # Count weights
    for name, param in model.named_parameters():
        if "weight" in name:
            total_weights += param.numel()
            zero_weights += (param == 0).sum().item()

    percent_pruned = 100 * zero_weights / total_weights

    # Evaluate accuracy
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)
    score = (val_accuracy/100 + zero_weights / total_weights ) / 2
    # Print summary
    print("Sparsity Report:")
    print(f"Total weights     : {total_weights:,}")
    print(f"Zeroed weights    : {zero_weights:,}")
    print(f"Percentage pruned : {percent_pruned:.2f}%")
    print(f"Final Val Accuracy: {val_accuracy:.2f}%")
    print(f"Final score: {score:.2f}")


In [125]:
report_sparsity_and_accuracy(model_no_bn, val_loader, criterion, device)

Sparsity Report:
Total weights     : 592,224
Zeroed weights    : 25,326
Percentage pruned : 4.28%
Final Val Accuracy: 60.44%
Final score: 0.32
